# 3 Loss, optimizer, and the loop

Notebook 01 gave you vectors, notebook 02 gave you the layer. This one closes the gap
between them: how a cache becomes batches, how you tell the model it is wrong, how it
changes its mind, and how you tell when it has learned enough.

None of this is new in 2026 — it is the machinery every course teaches. What is different
is that you already have something worth training on, so we can go through it once,
properly, at speed, and end with a **number that the rest of the lesson has to beat.**

That last part matters more than it sounds. Notebook 04 runs a search over dozens of
architectures. A search without a baseline established first is unreadable — you cannot
tell a win from a wobble.

In [ ]:
import sys
from pathlib import Path

import datasets
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache

from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

torch.manual_seed(42)

train_cache = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="train")
test_cache = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="test")

HIDDEN = train_cache.dim  # read the metadata, don't guess
N_CLASSES = train_cache.num_classes

print(f"{len(train_cache)} train / {len(test_cache)} test rows")
print(f"hidden_size {HIDDEN}, {N_CLASSES} classes")

## 3.1 From cache rows to batches

A cache row is a dict: one vector, a 1D tensor `(D)`, and one label, which is just the
number of the class, e.g. 5. A model wants a **batch**: one `(B, D)` 2D tensor of inputs,
e.g. with `B=32` there are 32 of those 1D tensors stacked on top of each other, and one
tensor of `B` targets. This is the `(D) → (B, D)` step
[01b_tensors.ipynb](01b_tensors.ipynb) §2 drew with five movie reviews: separate vectors,
stacked into one tensor. The function that does this is called a *collate function*, and
it is worth writing once by hand so it stops being magic.

In [ ]:
rows = [train_cache[i] for i in range(4)]
print("one row:", {k: tuple(v.shape) for k, v in rows[0].items()})


def collate_by_hand(rows: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    X = torch.stack([row["embed"] for row in rows])  # list of (D) -> (B, D)
    y = torch.stack([row["label"] for row in rows])  # list of ()  -> (B,)
    return X, y


X, y = collate_by_hand(rows)
print(f"\n{len(rows)} rows -> X {tuple(X.shape)}, y {tuple(y.shape)}")

In [ ]:
import matplotlib.pyplot as plt

from plotting import plot_stacking

class_names = train_cache.dataset.features["label"].names
labels = [
    f"cache row {i}: a tile of class {int(row['label'])}, '{class_names[int(row['label'])]}'"
    for i, row in enumerate(rows)
]
plot_stacking(
    [row["embed"] for row in rows],
    labels,
    kind="batch",
    title="Batching is stacking: four cache rows become one (B, D) tensor",
)
plt.show()

The same drawing as the batch of reviews in notebook 01b, now with EuroSAT tiles: four
independent items, four separate `(D)` vectors, one `torch.stack`. The labels go through
the same step on their own: four `()` scalars become one `(B,)` tensor.

That is the whole job: **a list of items becomes a pair of tensors.** Everything else a
collate function ever does is a variation on it — padding (adding zeros when you need to batch data with different sizes, eg sentences that are sometimes shorter, sometimes longer) when the items have different
lengths, stacking when they do not.

Ours is the easy case. `tensordtype: 1` in the metadata told us each item is `(D)`, so
`torch.stack` is enough. Lesson 5's chunked documents are `(C, D)` where every document is chunked into pieces of 512 tokens, giving a different `C` because the documents have different lengths, and *there* the padder has to do real work.

One more conversion happens on the way: the label. `train_cache["label"]` is a single
integer per row — class 5, say — but a classifier's job is not to name one class, it is to
output a **probability distribution over all of them**: "30% forest, 50% river, 20%
everything else." Training needs a target in that same shape to compare against, and
one-hot encoding is the simplest possible one: 100% on the true class, 0% everywhere else.
It is not the only choice — label smoothing spreads a little probability onto the other
classes on purpose, and lesson 5's documents can be genuinely multi-label — but single-class
one-hot is the right target for a task like this one, where exactly one class is correct.

### Three splits: train, valid, test

The cache comes with two splits, train and test. Training needs a third one. While you
train you will make decisions by looking at a split's score: when to stop, which model to
keep. Every decision like that fits your choices a little to that split, so its score ends
up a little too good. If you made them on test, the test score would no longer say how the
model does on data it has never been tuned for, and that is the only thing a test score is
for.

So the roles are:

| split | the model trains on it | you decide on it | you look at it |
|---|---|---|---|
| **train** | yes | no | every epoch |
| **valid** | no | yes: when to stop, which model wins | every epoch |
| **test** | no | no | once, at the end |

EuroSAT's cache has no valid split, so we carve one out of train: 20% of its rows, drawn
per class (`stratify_by_column="label"`) so every class keeps its share, with a fixed seed
so you get the same rows every time you run the notebook. Slide 43 shows the train and
valid curves this split makes possible.

The cell below carves the split and builds the three loaders with the library's
collate function, the same idea as `collate_by_hand` with the shapes checked.
`train_test_split` calls its two parts "train" and "test"; its "test" part is our
valid split, and the real test split is the cache's own.

In [ ]:
from vectormesh.data import Collate, OneHot

# CrossEntropyLoss accepts soft targets, so we one-hot the label into its own column
onehot = OneHot(num_classes=N_CLASSES, label_col="label", target_col="onehot")
collate_fn = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)

# 80% of the train rows to train on, 20% to decide on; test stays untouched
split = train_cache.dataset.train_test_split(
    test_size=0.2, seed=42, stratify_by_column="label"
)
trainset, validset = split["train"], split["test"]
testset = test_cache.dataset

BATCH = 64
trainloader = DataLoader(
    trainset.map(onehot), batch_size=BATCH, shuffle=True, collate_fn=collate_fn
)
validloader = DataLoader(
    validset.map(onehot), batch_size=BATCH, shuffle=False, collate_fn=collate_fn
)
testloader = DataLoader(
    testset.map(onehot), batch_size=BATCH, shuffle=False, collate_fn=collate_fn
)

X, y = next(iter(trainloader))
print(f"X {tuple(X.shape)}  {X.dtype}")
print(f"y {tuple(y.shape)}  {y.dtype}   <- one-hot, so (B, K) not (B,)")
print(f"{len(trainset)} train / {len(validset)} valid / {len(testset)} test rows")
print(
    f"{len(trainloader)} train batches, {len(validloader)} valid, {len(testloader)} test"
)

`(B, D) → (B, K)`. That is the entire trip from notebook 01's ladder, and you are now
standing on the second rung.

Note the loaders differ in one flag. `shuffle=True` on train, because a cache can be stored
grouped by label (you will meet one in notebook 05): without shuffling, a batch would hold
one class, and every step would pull the weights toward that class and away from the
others; `shuffle=False` on valid and test, because nothing is
learned there and you want a reproducible number.

## 3.2 Losses: what "wrong" means

A loss is a single number saying how badly the model did, differentiable so that we can ask
which direction reduces it. Picking one is not a detail — it *defines* the task.

| task | one item's target | loss | metric |
|---|---|---|---|
| regression | a real number | `MSELoss` / `L1Loss` | MAE, MASE |
| **single-label** classification | exactly one of K | **`CrossEntropyLoss`** | **Accuracy** |
| multi-label classification | any subset of K | `BCEWithLogitsLoss` | F1 |

**Our task is the middle row.** A EuroSAT tile is a forest or a river, not both, so the ten
scores should compete with each other — which is exactly what the softmax inside
`CrossEntropyLoss` does. Accuracy is the honest metric because the classes are roughly
balanced.

### Loss and metric are not the same thing, even when they look alike

A **loss** exists for the optimizer: it has to be differentiable everywhere it matters,
because `backward()` needs a gradient to follow. A **metric** exists for you: it has to be
interpretable, and it does not need a gradient at all — `Accuracy` is a step function
(right or wrong, nothing in between), which is exactly why it cannot be the loss.
`CrossEntropyLoss` is the smooth stand-in that *does* have a gradient and points in roughly
the direction accuracy would improve.

They share one requirement, though: **both need to define what "good" means for this task**,
and a metric can lie about that as easily as a loss can. A few you will use in this course,
beyond the classification pair above:

- **MAE** (mean absolute error) — a regression metric in the same units as the target, so
  "MAE of 2.3 degrees" means something a raw MSE (squared, unitless) does not. Also a valid
  loss (`L1Loss`) if you want the optimizer to weight all errors equally rather than
  squaring the large ones.
- **MASE** (mean absolute scaled error) — MAE divided by the MAE of the dumbest reasonable
  baseline (predict "same as last time"). `MASE < 1` means you beat that baseline;
  `MASE > 1` means a naive guess would have done better. Lesson 3's time-series models are
  judged this way, because a raw MAE number means nothing without knowing whether the
  series was easy or hard to predict in the first place.
- **F1** — the harmonic mean of precision and recall, used instead of accuracy when classes
  are imbalanced or when both false positives and false negatives cost something.
  Lesson 5's multi-label documents use it: a document can be tagged with several labels, so
  "percent exactly right" is far too strict a metric, and F1 rewards getting *most* of the
  labels right per document.

The pattern under all of them: a loss is chosen so the optimizer can follow it; a metric is
chosen so a human can trust it. Sometimes one formula does both jobs (MAE); often it takes
two (CrossEntropy + Accuracy, BCEWithLogits + F1).

Back to the table. The distinction to hold on to is *single-label vs multi-label*, and it
is about the data, not the model. The cell below shows it on three logits:

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])


def show(name, scores, comment):
    print(
        f"{name:24s} [{', '.join(f'{s:.3f}' for s in scores)}]   sums to {scores.sum():.3f}   {comment}"
    )


show("raw logits", logits, "")
show("softmax (CrossEntropy)", torch.softmax(logits, dim=0), "<- the classes compete")
show("sigmoid (BCEWithLogits)", torch.sigmoid(logits), "<- each class decided alone")

Raising one softmax score pushes the others down; raising one sigmoid score does nothing to
its neighbours. That is the whole difference, and it is why lesson 5's legal-document task
takes `BCEWithLogits` + F1: a ruling can be about tenancy law *and* liability at once, so
forcing those two labels to compete would be a modelling error, not a tuning choice.

Two practical notes that cost people hours:

- **`CrossEntropyLoss` eats logits, not probabilities.** Do not put a softmax at the end of
  your model — the loss applies its own, and doing it twice quietly flattens your gradients.
- **`BCEWithLogits` is `Sigmoid + BCELoss` fused**, and the fused version is the numerically
  stable one. Use it rather than assembling the pieces.

## 3.3 The optimizer, and the loop

Backpropagation gives you $\partial \text{loss} / \partial w$ for every weight: which way to
nudge each one. The optimizer decides how far to step. `Adam` adapts the step size per
parameter and is a fine default for everything in this course.

### What an optimizer does, drawn

A real model has thousands of weights, so its loss is a surface you cannot draw. Shrink the
model to two weights and you can. The surface below is a valley: steep along weight 2,
nearly flat along weight 1. Real networks have this shape too (some weights move the loss
a lot, others barely), and it is where optimizers differ.

Every run starts at the star and takes 40 steps toward the minimum at the cross. The loop
inside `descend` is the same `zero_grad → loss → backward → step` loop you will write for
the real model below; only the model is smaller.

In [ ]:
from plotting import plot_optimizer_paths


def valley(w: torch.Tensor) -> torch.Tensor:
    """A loss over two weights: steep along weight 2, nearly flat along weight 1."""
    return 0.1 * w[0] ** 2 + 2 * w[1] ** 2


def descend(make_optimizer, steps: int = 40) -> torch.Tensor:
    w = torch.tensor(
        [-4.0, 2.0], requires_grad=True
    )  # the two weights, at the starting point
    optimizer = make_optimizer([w])
    path = [w.detach().clone()]
    for _ in range(steps):
        optimizer.zero_grad()
        loss = valley(w)
        loss.backward()
        optimizer.step()
        path.append(w.detach().clone())
    return torch.stack(path)  # (steps + 1, 2): one row per step


fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plot_optimizer_paths(
    valley,
    {
        "SGD, lr=0.2": descend(lambda w: torch.optim.SGD(w, lr=0.2)),
        "SGD + momentum, lr=0.2": descend(
            lambda w: torch.optim.SGD(w, lr=0.2, momentum=0.7)
        ),
        "Adam, lr=0.3": descend(lambda w: torch.optim.Adam(w, lr=0.3)),
    },
    ax=axes[0],
    title="One valley, three optimizers",
)
plot_optimizer_paths(
    valley,
    {
        "SGD, lr=0.02": descend(lambda w: torch.optim.SGD(w, lr=0.02)),
        "SGD, lr=0.3": descend(lambda w: torch.optim.SGD(w, lr=0.3)),
        "SGD, lr=0.505": descend(lambda w: torch.optim.SGD(w, lr=0.505)),
    },
    ax=axes[1],
    title="SGD, three learning rates",
)
plt.tight_layout()
plt.show()

**Left, the optimizers.** Plain **SGD** steps along the gradient, times the learning
rate. It drops into the valley fast, because the steep direction has a large gradient,
and then crawls along the flat floor, where the gradient is tiny. **Momentum** adds a
running sum of earlier steps, so it picks up speed along the floor; the price is
overshooting, the swings across the valley. **Adam** divides each weight's step by the
recent size of that weight's own gradient, so both weights move at a similar pace whatever
the slope: that is why its first steps go diagonally. One learning rate then works for
weights whose gradients differ a lot in size, which is what makes Adam a good default. It
does not make Adam the winner everywhere: on this valley momentum gets closest to the
minimum (see the losses in the legend), and on another surface the order changes.

**Right, the learning rate.** The learning rate is the step size, and it has to fit the
surface. Too small (0.02): after 40 steps SGD has hardly moved. Too large (0.505): each
step overshoots the valley floor by more than the last, so the path bounces higher and the
loss grows instead of shrinking. In between (0.3) it settles into the valley and moves toward the minimum. On a real network you cannot
see the surface, only the loss curve: a loss that barely falls, or one that jumps around
and ends in `nan`, are these two failures.

### The loop

Here is the loop for the real model, by hand, once. Five lines, in this order, and getting
the order wrong is one of the classic silent bugs. Three epochs are enough to watch the
numbers move; how many epochs a model needs is the question of §3.4. After every epoch
the loop scores the valid split, the one you decide on.

```mermaid
flowchart LR
    A["1 zero_grad()<br/>clear old gradients"] --> B["2 forward: model(X)<br/>-> yhat"]
    B --> C["3 loss_fn(yhat, y)<br/>-> one number"]
    C --> D["4 loss.backward()<br/>gradient for every weight"]
    D --> E["5 optimizer.step()<br/>nudge the weights"]
    E -.next batch.-> A
```

Forward computes a prediction and a loss; backward turns that one number into a gradient
for every weight in the model, by the chain rule; `step()` uses those gradients to move the
weights. Nothing here is specific to this task — swap the model, the loss, or the data and
the five-line shape stays exactly the same.

In [ ]:
model = nn.Linear(HIDDEN, N_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(param(3, test=1)):
    total = 0.0
    for X, y in trainloader:
        optimizer.zero_grad()  # 1. clear last batch's gradients -- they ACCUMULATE
        yhat = model(X)  # 2. prediction (forward)
        loss = loss_fn(yhat, y)  # 3. how wrong was the prediction?
        loss.backward()  # 4. gradient of the loss w.r.t. every weight (backward)
        optimizer.step()  # 5. nudge the weights
        total += float(loss.detach())

    model.eval()
    with torch.no_grad():
        correct = sum(
            float((model(X).argmax(1) == y.argmax(1)).sum()) for X, y in validloader
        )
    model.train()
    print(
        f"epoch {epoch}  train loss {total / len(trainloader):.4f}  valid acc {correct / len(validset):.4f}"
    )

Things worth naming in that loop, because you will meet all of them as bugs:

- **`zero_grad()` first.** Gradients accumulate by default. Forget it and every step uses
  the sum of all gradients so far.
- **`no_grad()` for evaluation.** No gradients are needed, so torch does not keep
  what it would need to compute them.
- **`.eval()` / `.train()`.** Some layers behave differently in the two modes — dropout,
  batchnorm, and `GaussianNoise`, which you will meet in notebook 04. Evaluating in train
  mode gives you a noisy, irreproducible number.

That is the last time you will write this loop. It is short enough to understand and
repetitive enough that retyping it per experiment is how mistakes get in. From here on,
`vectormesh.training.Trainer` runs it, with the same five steps and the same
`.eval()`/`.train()`/`no_grad()` discipline handled for you.

The `train` function below wraps it. What it adds to the loop above:

- **an epoch is 15 steps here, not a whole pass.** `TrainerSettings.train_steps` sets how
  many batches the Trainer trains on before it evaluates; we set it to a fifth of the 75
  train batches. Five evaluations per pass over the train rows let you watch the curve
  develop instead of seeing one point per pass. In the rest of this notebook "epoch" means
  these 15 steps; `EPOCHS` = 80 of them is 16 passes over the train rows;
- after every epoch it computes the loss on the valid split, the *valid loss* (the Trainer
  calls it `test_loss`, whatever loader you hand it), and a small `record` function (a
  reporter: `Trainer` calls it once per epoch) keeps the train loss, the valid loss and the
  valid accuracy, so you can draw the curves;
- `epochs` is the largest number of epochs it may run, `EPOCHS`;
- `patience` switches early stopping on or off and `monitor` says which valid curve it
  watches; §3.5 explains both, until then early stopping is off.

`score` computes accuracy over every row of a loader. The valid accuracy is computed with
it, so valid and test accuracy are the same calculation. The test loader goes through it
once, in §3.6.

In [ ]:
from vectormesh.components import NeuralNet, Serial
from vectormesh.components.metrics import Accuracy
from vectormesh.data.vectorizers import detect_device
from vectormesh.training import Trainer, TrainerSettings

STEPS = len(trainloader) // 5  # one "epoch" = 15 batches: five evaluations per pass
EPOCHS = param(
    80, test=2
)  # the most epochs a run may take: 16 passes over the train rows
device = detect_device()


def score(model, loader) -> float:
    """Accuracy of model on every row of loader."""
    model.eval()
    with torch.no_grad():
        right = sum(
            float((model(X.to(device)).argmax(1) == y.to(device).argmax(1)).sum())
            for X, y in loader
        )
    return right / len(loader.dataset)


def train(
    model,
    patience: int | None = None,
    monitor: str = "loss",
    epochs: int = EPOCHS,
    trainloader=trainloader,
):
    """Train STEPS batches per epoch, evaluate on validloader after every epoch.

    patience=None runs every epoch; an int switches on early stopping, watching the
    valid loss (monitor="loss") or the valid accuracy (monitor="accuracy").
    Returns the Trainer's result and the curves per epoch.
    """
    early = None
    if patience is not None:
        early = {"patience": patience, "save": False, "verbose": False}
        if monitor == "accuracy":
            early |= {"monitor": "Accuracy", "mode": "max"}  # higher is better
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("modellogs").resolve(),
        train_steps=STEPS,
        valid_steps=len(validloader),
        earlystop_kwargs=early,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        device=device,
        progress=False,
    )
    curves = {"train_loss": [], "valid_loss": [], "valid_acc": []}

    def record(epoch, train_loss, valid_loss, metric_dict):
        curves["train_loss"].append(train_loss)
        curves["valid_loss"].append(valid_loss)  # the Trainer's test_loss
        curves["valid_acc"].append(score(trainer.model, validloader))

    trainer.reporters.append(record)
    result = trainer.loop()  # with early stopping: the best epoch, its weights restored
    return result, curves


print(f"training on {device}; {STEPS} steps per epoch, at most {EPOCHS} epochs")

## 3.4 How long to train: read the curve

How many epochs does a model need? Any number you type is a guess: too few and the model
had not finished, too many and it starts to memorise the training rows. You do not have to
guess, because the model tells you. Train for longer than it needs, and look at two curves
per epoch: the **train loss** (on the rows it learns from) and the **valid loss** (on rows
it never learns from).

Three shapes to recognise, the same ones slide 43 shows:

- **still learning**: both curves are still falling; more epochs would help;
- **finished**: both curves are flat; more epochs only cost time;
- **overfitting**: the train loss keeps falling while the valid loss turns up; the model is
  fitting the training rows in a way that no longer carries over to new ones.

The loss is not the only thing to look at. Next to it the cell draws the **valid
accuracy**, the number you will report, because a loss and a metric can tell different
stories (§3.2).

**The model.** A narrow head with two hidden layers, `NeuralNet([384, 64, 64], 10)`:

`(B, 384) → Linear 384→64 → GELU → Linear 64→64 → GELU → Linear 64→10 → (B, 10)`

The first `Linear` squeezes the 384 numbers of a DINOv2 vector into 64, the second mixes
those 64, the last turns them into one logit per class. We use a narrow head on purpose: a
head as wide as the input, `NeuralNet(384, 10)`, has many more weights to fit the train
rows with and learns them fast, so most of its curve is over within the first
evaluations. With 64 units the head learns slower, and together with the 15-step epochs the
curve shows learning, the best point and what comes after. It is not chosen because it
scores better; E6 lets you look at the wide head's curve. `make_head` builds it, here and in §3.6.

It trains for all `EPOCHS` = 80 epochs, early stopping off. Read the curve once it has
run: which of the three shapes does it end in, and at which epoch would you have stopped
it?

The curve is one run with one seed. Its exact epochs move with the seed and the machine;
the shape is what to read.

In [ ]:
import numpy as np


def make_head():
    return Serial([NeuralNet(hidden_size=[HIDDEN, 64, 64], out_size=N_CLASSES)])


torch.manual_seed(42)
_, full = train(make_head())

lowest = int(np.argmin(full["valid_loss"]))
top = int(np.argmax(full["valid_acc"]))
print(
    f"{len(full['valid_loss'])} epochs of {STEPS} steps, numbered 0 to {len(full['valid_loss']) - 1}"
)
print(
    f"valid loss: lowest {full['valid_loss'][lowest]:.4f} at epoch {lowest}, last {full['valid_loss'][-1]:.4f}"
)
print(
    f"valid acc:  {full['valid_acc'][lowest]:.4f} at epoch {lowest}, highest {full['valid_acc'][top]:.4f} at epoch {top}, last {full['valid_acc'][-1]:.4f}"
)
print(
    f"train loss: {full['train_loss'][0]:.3f} at epoch 0, {full['train_loss'][lowest]:.3f} at epoch {lowest}, {full['train_loss'][-1]:.3f} last"
)
print(
    "valid loss every 5th epoch:", " ".join(f"{v:.3f}" for v in full["valid_loss"][::5])
)
print(
    "valid acc  every 5th epoch:", " ".join(f"{v:.3f}" for v in full["valid_acc"][::5])
)

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
ax.plot(full["train_loss"], color="tab:blue", label="train loss")
ax.plot(full["valid_loss"], color="tab:orange", label="valid loss")
ax.set_ylim(0, 0.6)
ax.set_ylabel("cross-entropy loss")
ax.set_title(f"Valid loss falls, is lowest at epoch {lowest}, then creeps up")
ax.legend(frameon=False)
ax2.plot(full["valid_acc"], color="tab:orange")
ax2.set_ylim(0.85, 0.96)
ax2.set_ylabel("valid accuracy (fraction of 1200 rows)")
ax2.set_title("Valid accuracy over the same epochs")
for a in (ax, ax2):
    a.set_xlabel(f"epoch ({STEPS} steps; 5 epochs = one pass over the train rows)")
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

All three shapes are in this one run.

- **Learning**, epochs 0 to about 20: the valid loss falls from 1.06 to 0.18 and the valid
  accuracy climbs from 0.72 to 0.93.
- **About finished**, epochs 20 to about 45: the valid loss wobbles between 0.17 and 0.20,
  lowest at **epoch 29** (0.170), while the train loss keeps falling (0.067 at epoch 29).
- **Overfitting**, after about epoch 50: the valid loss creeps up to 0.24 at the end, while
  the train loss falls to 0.010.

The accuracy tells a milder story. From about epoch 25 the valid accuracy stays between
0.931 and 0.946 and does not fall with the loss: it is highest at epoch 30 (0.946) and
0.940 at the end. So the rising valid loss is not more rows wrong. Cross-entropy charges a
wrong answer by how confident it was, so the loss can rise while the accuracy holds: the
model grows more confident on the rows it still gets wrong. Overfitting shows up in the
loss as overconfidence first, and within these 80 epochs it hardly costs accuracy.

## 3.5 Early stopping: let the valid curve decide

The curve already told you when to stop. Early stopping is that reading done by the loop
itself, every epoch, so no epoch count has to be picked by hand:

1. after each epoch, compute the valid loss;
2. if it is the lowest so far, copy the model's weights and set a counter to 0;
3. if it is not, add 1 to the counter;
4. when the counter reaches the **patience**, stop, and put back the copied weights.

`epochs` then stops being *the* number of epochs and becomes the most a run may take. You
get back the weights of the best epoch, not the last one.

Why not stop at the first epoch that is worse? Because the valid curve wobbles: 1200 rows
give a noisy number, and a bad epoch can be followed by a new best. The patience is how many
epochs without a new best you accept before you believe the curve has turned. Too short and
a wobble stops you; too long and you pay for epochs you throw away. `PATIENCE` = 10 epochs
is two passes over the train rows.

**Which curve to watch is your decision.** The valid loss is the usual default, and the
Trainer's: it moves smoothly, because every probability the model gives changes it, while
accuracy only moves when a row flips from wrong to right, in steps of 1/1200 here. And the
loss is the one number every task has. But §3.4 showed that on this task the loss and the
accuracy disagree once the model grows overconfident, and the accuracy is what you will
report. Do not use early stopping blindly: look at both curves, ask what you actually care
about, and stop on that.

`where_it_stops` applies the four steps to a curve you already have, so you can see where a
run would stop without training it again. With `mode="max"` it looks for the highest
value instead of the lowest, so it works on the valid accuracy too. The cell runs it on both
curves of §3.4.

In [ ]:
def where_it_stops(
    curve: list[float], patience: int, mode: str = "min"
) -> tuple[int, int]:
    """(best epoch, last epoch that runs) under early stopping with this patience.

    mode="min" for a loss (lower is better), "max" for an accuracy (higher is better).
    """
    better = (lambda a, b: a < b) if mode == "min" else (lambda a, b: a > b)
    best, counter = 0, 0
    for epoch, value in enumerate(curve):
        if better(value, curve[best]):
            best, counter = epoch, 0
        elif epoch > 0:
            counter += 1
            if counter == patience:
                return best, epoch
    return best, len(curve) - 1  # never ran out of patience: the budget ended it


PATIENCE = 10  # two passes over the train rows: longer than a wobble, short enough to waste little

on_loss = where_it_stops(full["valid_loss"], PATIENCE, mode="min")
on_acc = where_it_stops(full["valid_acc"], PATIENCE, mode="max")
lowest = int(np.argmin(full["valid_loss"]))
print(
    f"lowest valid loss of the whole run: epoch {lowest} ({full['valid_loss'][lowest]:.4f})"
)
for name, (best, stop) in [("valid loss", on_loss), ("valid accuracy", on_acc)]:
    print(
        f"stop on {name:14s}: keeps epoch {best} (valid acc {full['valid_acc'][best]:.4f}, "
        f"valid loss {full['valid_loss'][best]:.4f}), stops after epoch {stop}"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
panels = [
    ("valid_loss", on_loss, "cross-entropy loss", (0, 0.6), "on the valid loss"),
    ("valid_acc", on_acc, "valid accuracy", (0.85, 0.96), "on the valid accuracy"),
]
for ax, (key, (best, stop), ylabel, ylim, label) in zip(axes, panels):
    ax.plot(full[key], color="tab:orange")
    if key == "valid_loss":
        ax.plot(full["train_loss"], color="tab:blue", label="train loss")
    ax.axvspan(
        best + 0.5,
        stop + 0.5,
        color="tab:orange",
        alpha=0.12,
        label=f"{PATIENCE} epochs of patience",
    )
    ax.axvspan(
        stop + 0.5, len(full[key]) - 0.5, color="grey", alpha=0.12, label="never run"
    )
    ax.scatter(
        [best],
        [full[key][best]],
        color="tab:red",
        zorder=3,
        s=60,
        label=f"kept: epoch {best}",
    )
    ax.axvline(
        stop + 0.5, color="black", lw=1, ls="--", label=f"stop after epoch {stop}"
    )
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(f"epoch ({STEPS} steps)")
    ax.set_title(f"Early stopping {label}, patience {PATIENCE}")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(
        frameon=False,
        fontsize=8,
        loc="lower right" if key == "valid_acc" else "upper right",
    )
plt.tight_layout()
plt.show()

The two choices disagree here.

- **On the valid loss** the run keeps epoch 17 (valid accuracy 0.936) and stops after epoch
  27.
- **On the valid accuracy** it keeps epoch 30 (valid accuracy 0.946) and stops after epoch 40.

That is one point of valid accuracy, 12 of 1200 rows, in favour of stopping on accuracy.
Read it with care: epoch 30 is the highest point of an accuracy curve that wobbles between
0.931 and 0.946 from epoch 25 on, and the highest of many noisy values is a little lucky
(notebook 05 comes back to this). What the picture does show is that stopping on the loss
stopped while the accuracy was still climbing, and that the accuracy did not turn down
afterwards. For a model you will be judged on by accuracy, that is the curve to stop on
here.

Neither choice keeps the last epoch's weights, and both stop long before epoch 79. The curve
is one seed; with another seed the epochs move, and so can the size of the disagreement.

### Do not stop on one number without looking

Put two lines of that output side by side. The lowest valid loss of the whole run is at
**epoch 29** (0.170). Stopping on the valid loss kept **epoch 17** (0.176) and stopped after
**epoch 27**, two epochs before that low. The rule did what it says: epochs 18 to 27 brought
no new low, ten in a row, so it called the curve turned. But the left picture shows no turn
there. From about epoch 20 to 45 the valid loss lies on a plateau and wobbles between 0.17
and 0.20. Ten epochs without a new low on a wobbly plateau do not mean the curve is going up.

Measured in accuracy the miss is small here (0.936 at epoch 17, 0.938 at epoch 29). The
lesson is about the rule, not this gap. Early stopping turns one curve into one decision, and
it cannot see the epochs it did not run. Before you trust where it stopped, look at the
recorded curve. Does it turn up after the kept epoch, or does it run on a plateau? Is the
patience longer than the wobble you see? And is the curve it watched the one you care about?
E5 asks how long the patience must be to reach epoch 29.

## 3.6 The baseline

Now the number. The baseline is `make_head()`, the narrow head of §3.4. `Serial` chains
components; with a single component it does nothing at all, and it is written this way
because notebook 04 will add things to that list.

§3.5 is the decision of what to stop on: this head is judged by accuracy, and on this task
its valid loss turns up from overconfidence while its accuracy holds. So the baseline stops
on the valid accuracy, `monitor="accuracy"`, with `PATIENCE` = 10, at most `EPOCHS` = 80.
To see what that choice changes, the cell also trains the same head stopping on the valid
loss, and compares the two **on valid**: valid is where you decide. Only the chosen one
goes to the test split, once. Looking at test for both and keeping the better would make
test a split you decide on.

**Predict first, from the pictures above:** for each run, which epoch will it keep, and
after which epoch will it stop?

This is the model to beat.

In [ ]:
runs = {}
for monitor in ["loss", "accuracy"]:
    torch.manual_seed(42)
    model = make_head()
    result, _ = train(model, patience=PATIENCE, monitor=monitor)
    runs[monitor] = (model, result)
    print(
        f"stop on valid {monitor:8s}: keeps epoch {result.epoch:2d}, stopped after epoch "
        f"{result.stopped_epoch:2d} ({result.stopped_epoch + 1} of at most {EPOCHS} epochs), "
        f"valid loss {result.test_loss:.4f} (the Trainer's test_loss), "
        f"valid accuracy {score(model, validloader):.4f}"
    )

baseline, result = runs["accuracy"]  # the choice of §3.5, made on valid
params = sum(p.numel() for p in baseline.parameters() if p.requires_grad)
test_accuracy = score(baseline, testloader)  # the one look at test
print(
    f"\nBASELINE (stopped on valid accuracy)  test accuracy {test_accuracy:.4f}   {params:,} parameters"
)

Both runs stop exactly where `where_it_stops` predicted from the curves of §3.4:

- stopping on the valid loss keeps epoch 17 and stops after epoch 27, with 28 epochs trained
  and a valid accuracy of 0.936;
- stopping on the valid accuracy keeps epoch 30 and stops after epoch 40, with 41 epochs
  trained and a valid accuracy of 0.946.

Choosing on valid, the baseline is the accuracy-stopped head. Write down its number together
with how it was trained: **test accuracy 0.951**, `make_head()` with 29,450 parameters,
stopped on the valid accuracy with patience 10, weights of epoch 30. The test split was
looked at once, for this model only. It lands half a point above the valid accuracy. Valid
and test are different rows, so the two need not match.

**It is cheap.** Not because the problem is easy, but because you are training over cached
vectors instead of images. The same accuracy from pixels means a CNN, a GPU, and a coffee
break. That asymmetry is the entire reason this course is arranged the way it is.

**It leaves few points to win.** At 0.95 on test, a better head on these vectors can win at
most five points, and probably far fewer.

**It is a small model.** Two narrow hidden layers, no fusion, no gating, no tuning. Every
architecture you are about to try has to earn the difference, and the honest default
expectation is that most of them will not.

That is not cynicism. It is the reason the search is worth running: you find out instead of
arguing about it. Notebook 04 does exactly that, on DTD, a texture dataset with 47 classes.

## What you now know

- A collate function turns a list of items into `(X, y)`. Stacking is enough for `(D)`
  items; padding is for the chunked case.
- The loss defines the task: `CrossEntropy` when classes compete, `BCEWithLogits` when they
  do not. Feed both of them logits.
- The optimizer decides the step. The learning rate has to fit the loss surface: too small
  crawls, too large diverges. Momentum and Adam help along flat directions.
- The loop is `zero_grad → forward → loss → backward → step`, and `eval()`/`no_grad()`
  around evaluation.
- Three splits: train to learn from, valid to decide on, test to look at once. A split you
  decide on gives a score that is a little too good, so it cannot be the one you report.
- A loss curve says whether a model is still learning (both falling), finished (both
  flat) or overfitting (train falling, valid rising). Read the valid accuracy next to it:
  the valid loss can rise from overconfidence while the accuracy holds.
- Early stopping reads a valid curve for you: it keeps the weights of the best epoch and
  stops after `patience` epochs without a new best. `epochs` becomes a maximum; the
  patience has to outlast a wobble of the curve.
- What it watches is your decision. The valid loss is the default; when you will be judged
  on a metric and the loss and the metric disagree, stop on the metric. Look at the curves
  before you trust any automatic stop.
- You have a baseline on EuroSAT, established *before* any searching, with its best and
  stopped epoch next to its score. That is what makes the next notebook readable.

## Exercises

**E1 — the wrong loss on purpose.** Swap `CrossEntropyLoss` for `BCEWithLogitsLoss` on this
single-label task and retrain. Accuracy barely moves. Explain why the metric fails to
notice, and what would have to be true about EuroSAT for the swap to be *correct* rather
than merely survivable.

**E2 — where does the gap come from?** Shrink the training rows to 100
(`trainset.select(range(100))`), build a loader for them the way §3.1 does, and retrain
with `train(make_head(), patience=PATIENCE, monitor="accuracy", trainloader=small_loader)`.
Which moves more: the training loss, or the gap between train and valid loss? And does
early stopping stop earlier or later than the baseline of §3.6? Predict before running,
then say which of the numbers tells you that you are overfitting.

**E3 — skip a line.** Delete `optimizer.zero_grad()` from the hand-written loop in §3.3 and
run it. Describe what the loss curve does and explain it in terms of accumulated gradients.
(This is the single most common torch bug; meeting it deliberately once is cheaper than
meeting it accidentally at 2am.)

**E4 — is your baseline stable?** Rerun the accuracy-stopped baseline of §3.6 with five
different seeds. For each, print the kept epoch, the stopped epoch and the valid accuracy
(`score(model, validloader)`), then the mean and std of the valid accuracy: decisions are
made on valid, so that is the spread a decision has to beat. Is the difference §3.6 found
between stopping on the loss and stopping on the accuracy larger than that spread?

**E5 — change the patience.** Before running anything, read the curves of §3.4 and write
down, for a patience of 3, 10 and 30, which epoch each choice keeps and after which epoch it
stops, on the loss and on the accuracy. Check your reading with
`where_it_stops(full["valid_loss"], p)` and `where_it_stops(full["valid_acc"], p, mode="max")`.
Which patience lets the loss-stopped run reach its lowest point at epoch 29? Which spends
the most epochs? What would you need to see in a curve to choose a patience longer than 10?

**E6 — the wide head.** Run §3.4 with `NeuralNet(HIDDEN, N_CLASSES)`, one hidden layer as
wide as the input, instead of `make_head()`. Where is its lowest valid loss, and how many
epochs of the curve show it still learning? Why does a head with more weights fit the train
rows sooner?